# Redrob Candidate Discovery & Ranking — Data Exploration

This notebook documents the Exploratory Data Analysis (EDA) on the `candidates.jsonl` dataset (100,000 candidate profiles) and `sample_candidates.json` (50 candidate profiles).

## 1. Schema Analysis
The candidates pool has the following top-level structure per candidate:
- `candidate_id`: Format `CAND_XXXXXXX` (7 digits)
- `profile`: Basic metadata (headline, summary, years of experience, current title/company)
- `career_history`: Array of job roles (dates, durations, is_current, description)
- `education`: Academic history (degree, institution, tier)
- `skills`: Skill names, self-reported proficiency, and duration
- `redrob_signals`: Behavioral signals (active date, response rate, notice period, expected salary, github score, etc.)

## 2. Industry Distribution Analysis
Scanning all 100,000 candidate profiles reveals the following distribution of `current_industry` values. This is used to build the Services-vs-Product classification heuristic:

| Current Industry | Candidate Count | Percentage (%) |
| :--- | :--- | :--- |
| **IT Services** | 29,881 | 29.88% |
| **Software** | 22,417 | 22.42% |
| **Manufacturing** | 22,305 | 22.30% |
| **Conglomerate** | 7,571 | 7.57% |
| **Paper Products** | 7,467 | 7.47% |
| **Fintech** | 2,808 | 2.81% |
| **Food Delivery** | 2,514 | 2.51% |
| **E-commerce** | 1,529 | 1.53% |
| **Consulting** | 1,274 | 1.27% |
| **EdTech** | 610 | 0.61% |
| **SaaS** | 328 | 0.33% |
| **AI/ML** | 278 | 0.28% |
| **AdTech** | 172 | 0.17% |
| **Other** | 1,348 | 1.35% |

IT Services and Consulting combined represent **31.15%** of the candidate pool. Candidates whose entire career is spent at these firms will be heavily penalized.

## 3. Honeypot Discovery & Anomaly Verification
The dataset contains subtly impossible profiles designed to test whether the system actually parses descriptions and validates dates, rather than just matching buzzwords.

We discovered exactly **55 unique honeypots** by running three strict checks:
1. **Expert Zero-duration**: Claims `expert` proficiency on 5+ skills with exactly `0` months duration.
2. **Date Mismatch**: Reported `duration_months` in a career role deviates from the actual start/end dates difference by > 2 months.
3. **YOE Mismatch**: Stated `years_of_experience` deviates from the sum of career history role durations by > 3 years.

Let's look at the script used to compile this union of honeypot profiles.

In [ ]:
# Code used to identify and verify the 55 honeypot candidates
import json
from datetime import datetime

candidates_path = '../candidates.jsonl'
current_date = datetime(2026, 6, 20)

expert_zero_set = set()
date_mismatch_set = set()
yoe_mismatch_set = set()

with open(candidates_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        c = json.loads(line)
        cid = c.get('candidate_id')
        p = c.get('profile', {})
        career = c.get('career_history', {})
        skills = c.get('skills', [])
        yoe = p.get('years_of_experience', 0)
        
5        # Check 1: Expert Zero duration
        exp_zero = [s for s in skills if s.get('proficiency') == 'expert' and s.get('duration_months') == 0]
        if len(exp_zero) >= 5:
            expert_zero_set.add(cid)
            
        # Check 2: Date vs reported duration mismatch
        has_date_mismatch = False
        for r in career:
            sd = r.get('start_date')
            ed = r.get('end_date')
            dm = r.get('duration_months', 0)
            if sd:
                s_dt = datetime.strptime(sd, "%Y-%m-%d")
                e_dt = current_date if r.get('is_current') else datetime.strptime(ed, "%Y-%m-%d") if ed else None
                if e_dt:
                    calc_dm = (e_dt.year - s_dt.year) * 12 + (e_dt.month - s_dt.month)
                    if abs(calc_dm - dm) > 2:
                        has_date_mismatch = True
        if has_date_mismatch:
            date_mismatch_set.add(cid)
            
        # Check 3: YOE sum mismatch
        sum_duration_years = sum(r.get('duration_months', 0) for r in career) / 12.0
        if abs(yoe - sum_duration_years) > 3.0 and sum_duration_years > 0:
            yoe_mismatch_set.add(cid)

union_set = expert_zero_set.union(date_mismatch_set).union(yoe_mismatch_set)
print(f"Expert Zero Count: {len(expert_zero_set)}")
print(f"Date Mismatch Count: {len(date_mismatch_set)}")
print(f"YOE Mismatch Count: {len(yoe_mismatch_set)}")
print(f"Total Unique Honeypot Candidates Discovered: {len(union_set)}")

## 4. Conclusion & Scoring Strategy
- Honeypots are hard-excluded from our top 100 ranking list (their scores are zeroed out).
- IT services-only profiles receive a heavy 0.5x penalty.
- Title-substance mismatches are filtered out.
- Semantic similarity is computed on the remaining subset to rank optimal candidates.